# 03 — Pipeline de transformation

Benoit Girard

L'étape *Transform* : passer de publications brutes hétérogènes à un jeu de données propre, typé et conforme au schéma. Le pipeline est organisé en trois temps — **lecture**, **traitement**, **export** — et chaque transformation unitaire est une petite fonction testable.

In [1]:
import sys
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE / "src"))

from dotenv import load_dotenv

load_dotenv(RACINE / ".env")

from multimodal_etl.logging_setup import setup_logging

setup_logging()

## 1. Les fonctions unitaires

Chacune fait une seule chose, ce qui les rend lisibles et testables séparément (`tests/test_transform.py`).

In [2]:
from multimodal_etl.transform import (
    extrait_domaine,
    nettoie_texte,
    normalise_date,
    normalise_label,
)

print(nettoie_texte("<p>Un   résumé &amp; son <b>HTML</b></p>"))
print(extrait_domaine("https://www.bbc.co.uk/news/article-123"))
print(normalise_date("Mon, 29 Jun 2026 10:00:00 GMT"))
print(normalise_date("1767225600"))
print(normalise_label("FAKE"), normalise_label("Real"), normalise_label(""))

Un résumé & son HTML
bbc.co.uk
2026-06-29T10:00:00+00:00
2026-01-01T00:00:00+00:00
fake real None


Les dates méritent un mot : chaque source a son format — RFC 822 pour le RSS, ISO pour l'API, horodatage Unix pour Fakeddit. Sans normalisation, l'indicateur de fraîcheur serait tout simplement incalculable.

## 2. La règle qui définit le jeu de données

`valide_image` ne regarde pas l'URL : elle vérifie que **le fichier est sur le disque**. Une publication sans image n'entre pas dans le jeu de données. C'est ce contrôle qui garantit l'association texte-image exigée par le cas d'usage.

In [3]:
from multimodal_etl.transform import valide_image

print(valide_image("data/raw/images/inexistante.jpg"))
print(valide_image(""))

False
False


## 3. Lecture → traitement → export

In [4]:
from multimodal_etl.config import RAW_DIR, TransformConfig
from multimodal_etl.transform import exporte, lit_brut, traite

config = TransformConfig()
dernier_brut = sorted(RAW_DIR.glob("raw_publications_*.json"))[-1]
brut = lit_brut(dernier_brut)
print(f"{len(brut)} publications brutes lues")

2026-08-20 12:03:10 | INFO    | multimodal_etl.transform | Transformation : lecture du fichier brut C:\Users\Ben\Desktop\multimodal_etl\data\raw\raw_publications_20260820_100255.json


2026-08-20 12:03:10 | INFO    | multimodal_etl.transform | Transformation : 179 publications brutes lues


179 publications brutes lues


In [5]:
df, stats = traite(brut, config)
stats

2026-08-20 12:03:10 | INFO    | multimodal_etl.transform | Transformation : 131/179 publications valides après nettoyage


2026-08-20 12:03:10 | INFO    | multimodal_etl.transform | Transformation : 0 doublons retirés


{'total_brut': 179,
 'total_valide': 131,
 'rejetes': 48,
 'doublons': 0,
 'avec_image': 131,
 'labellisees': 27,
 'images_natives': 124,
 'images_open_graph': 7}

Le taux de rejet est élevé, et c'est normal : presque toutes les publications écartées le sont pour une raison unique — aucune image exploitable. On le vérifie.

In [6]:
from multimodal_etl.transform import construit_publication

raisons = {"titre vide": 0, "texte trop court": 0, "pas d'image": 0, "retenue": 0}
for publication in brut:
    if not nettoie_texte(str(publication.get("title", ""))):
        raisons["titre vide"] += 1
    elif len(nettoie_texte(str(publication.get("text", "")))) < config.min_text_length:
        raisons["texte trop court"] += 1
    elif not valide_image(str(publication.get("image_path", ""))):
        raisons["pas d'image"] += 1
    else:
        raisons["retenue"] += 1
raisons

{'titre vide': 0, 'texte trop court': 5, "pas d'image": 43, 'retenue': 131}

## 4. Le jeu de données produit

Les colonnes viennent directement de `multimodal_etl.schema`, source unique de vérité partagée par le code, le diagramme et la documentation.

In [7]:
df[["source", "access_method", "title", "image_source", "has_image", "label"]].head(8)

,source,access_method,title,image_source,has_image,label
0,rss:the_guardian,flux_rss,More than 100 dead after goldmine collapses in...,native,True,NaN
1,rss:the_guardian,flux_rss,Five Americans among seven killed in safari he...,native,True,NaN
2,rss:the_guardian,flux_rss,Spain to allow 500 children in Ceuta to go to ...,native,True,NaN
3,rss:the_guardian,flux_rss,Ebola outbreak in Democratic Republic of the C...,native,True,NaN
4,rss:the_guardian,flux_rss,Zimbabwe boat accident death toll hits 68 as 2...,native,True,NaN
5,rss:the_guardian,flux_rss,Adviser to far-right Latin American leaders ar...,native,True,NaN
6,rss:the_guardian,flux_rss,Activist forced out of US after criticising Tr...,native,True,NaN
7,rss:the_guardian,flux_rss,"Trump pauses Canada tariffs threat, and hints ...",native,True,NaN


In [8]:
print("Répartition par méthode d'accès :")
print(df["access_method"].value_counts().to_string())
print()
print("Origine des images :")
print(df["image_source"].value_counts().to_string())

Répartition par méthode d'accès :
access_method
flux_rss                 97
telechargement_kaggle    20
api_rest                  7
telechargement_github     7

Origine des images :
image_source
native        124
open_graph      7


## 5. Export

Le format retenu est le **Parquet** : colonnaire, typé, compact. À ce stade le schéma est fixe et le fichier est destiné à de la lecture analytique. Les statistiques sont écrites à côté, sous le même nom : le tableau de bord charge toujours la paire, jamais un jeu de données orphelin.

In [9]:
chemin = exporte(df, config, stats)
print("Dataset :", chemin.name)
print("Statistiques :", chemin.with_name(chemin.stem + "_stats.json").name)

2026-08-20 12:03:11 | INFO    | multimodal_etl.transform | Transformation : dataset de 131 lignes exporté vers C:\Users\Ben\Desktop\multimodal_etl\data\processed\publications_20260820_100311.parquet


2026-08-20 12:03:11 | INFO    | multimodal_etl.transform | Transformation : statistiques écrites dans C:\Users\Ben\Desktop\multimodal_etl\data\processed\publications_20260820_100311_stats.json


Dataset : publications_20260820_100311.parquet
Statistiques : publications_20260820_100311_stats.json
